# 🔤 Hindi Ambiguity Parser — Transformer (mT5-small)
## NLP Assignment: Discuss Ambiguity in NLP & Train a Transformer on Hindi

---

### 📌 Project Flow
```
Input Hindi Sentence
        │
        ▼
┌──────────────────────┐
│  Ambiguity Detected  │   ← "यह वाक्य अस्पष्ट है!"
└──────────────────────┘
        │
   ┌────┴────┐
   ▼         ▼
अर्थ 1     अर्थ 2
(Meaning 1) (Meaning 2)
```

### 🧠 Types of Ambiguity Covered
| Type | Hindi Name | Example |
|------|-----------|---------|
| Lexical | शाब्दिक | "सोना" = gold OR sleeping |
| Syntactic | वाक्यात्मक | "मैंने उड़ते पक्षी को देखा" |
| Semantic | अर्थपरक | "वह चला गया" = left OR died |
| Pragmatic | व्यावहारिक | "यहाँ ठंड है" = fact OR request |

**Model**: `google/mt5-small` ≈ **300 Million parameters** (well under 2B limit)


In [1]:
# Install all required packages
!pip install transformers datasets torch sentencepiece accelerate ipywidgets -q

import importlib, sys
for pkg in ["transformers","datasets","torch","sentencepiece"]:
    v = importlib.import_module(pkg).__version__ if pkg != "sentencepiece" else "ok"
    print(f"  ✅ {pkg}: {v}")
print("\n🎉 All packages ready!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.8 MB/s eta 0:00:00
  ✅ transformers: 5.0.0
  ✅ datasets: 4.0.0
  ✅ torch: 2.10.0+cu128
  ✅ sentencepiece: ok

🎉 All packages ready!


In [2]:
import torch
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from transformers import (
    AutoTokenizer,
    MT5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from datasets import Dataset
from IPython.display import display, HTML

# ── Device setup ─────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔧 PyTorch  : {torch.__version__}")
print(f"💻 Device   : {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU      : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found — training will be slower on CPU.")
    print("   Go to  Runtime → Change runtime type → GPU  for speed.")


🔧 PyTorch  : 2.10.0+cu128
💻 Device   : cuda
🎮 GPU      : Tesla T4


## 📝 Step 3: Ambiguous Hindi Sentence Dataset

Each entry has:
- **hindi_sentence** — the ambiguous input
- **ambiguity_type** — category of ambiguity
- **ambiguous_word** — the word / phrase causing ambiguity
- **interpretation_1** & **interpretation_2** — the two possible meanings


In [3]:
# ── 25 handcrafted ambiguous Hindi sentences ────────────────────
RAW_DATA = [

    # ───── Lexical Ambiguity (शाब्दिक अस्पष्टता) ─────────────────
    {
        "hindi_sentence": "मैंने उसे मारा।",
        "ambiguity_type": "शाब्दिक (Lexical)",
        "ambiguous_word": "मारा",
        "interpretation_1": "मैंने उसे मुक्का मारा। (I hit / punched him.)",
        "interpretation_2": "मैंने उसे जान से मारा। (I killed him.)",
    },
    {
        "hindi_sentence": "वह बैंक गया।",
        "ambiguity_type": "शाब्दिक (Lexical)",
        "ambiguous_word": "बैंक",
        "interpretation_1": "वह नदी के किनारे (बैंक) गया। (He went to the riverbank.)",
        "interpretation_2": "वह पैसे जमा करने बैंक गया। (He went to the financial bank.)",
    },
    {
        "hindi_sentence": "सोना अच्छा है।",
        "ambiguity_type": "शाब्दिक (Lexical)",
        "ambiguous_word": "सोना",
        "interpretation_1": "सोना (स्वर्ण) बहुत कीमती है। Gold is precious.",
        "interpretation_2": "सोना (नींद लेना) स्वास्थ्य के लिए अच्छा है। Sleeping is good for health.",
    },
    {
        "hindi_sentence": "आम आदमी खुश है।",
        "ambiguity_type": "शाब्दिक (Lexical)",
        "ambiguous_word": "आम",
        "interpretation_1": "एक साधारण / सामान्य व्यक्ति खुश है। (A common/ordinary man is happy.)",
        "interpretation_2": "आम (फल) बेचने वाला आदमी खुश है। (The mango seller is happy.)",
    },
    {
        "hindi_sentence": "वह ठंडा इंसान है।",
        "ambiguity_type": "शाब्दिक (Lexical)",
        "ambiguous_word": "ठंडा",
        "interpretation_1": "उसके शरीर का तापमान कम है। (His body temperature is physically low.)",
        "interpretation_2": "उसका स्वभाव उदासीन और भावनाहीन है। (He has a cold / indifferent personality.)",
    },
    {
        "hindi_sentence": "काला पानी बहुत कठिन था।",
        "ambiguity_type": "शाब्दिक (Lexical)",
        "ambiguous_word": "काला पानी",
        "interpretation_1": "काले रंग का पानी पीना बहुत कठिन था। (Drinking black-colored water was very difficult.)",
        "interpretation_2": "काला पानी (अंडमान जेल) की सज़ा बहुत कठिन थी। (The Kala Pani prison punishment was very harsh.)",
    },
    {
        "hindi_sentence": "वह चला गया।",
        "ambiguity_type": "शाब्दिक (Lexical)",
        "ambiguous_word": "चला गया",
        "interpretation_1": "वह वहाँ से चलकर दूसरी जगह चला गया। (He walked away to another place.)",
        "interpretation_2": "वह इस दुनिया से चला गया। (He passed away / died.)",
    },
    {
        "hindi_sentence": "उसकी आँखें बोलती हैं।",
        "ambiguity_type": "शाब्दिक (Lexical)",
        "ambiguous_word": "बोलती हैं",
        "interpretation_1": "उसकी आँखें बहुत भाव व्यक्त करती हैं। (Her eyes express a lot of emotions.)",
        "interpretation_2": "उसकी आँखों में जादू है जो सचमुच बोलती हैं। (Her eyes have magic — they literally speak.)",
    },
    {
        "hindi_sentence": "दिल टूट गया।",
        "ambiguity_type": "शाब्दिक (Lexical)",
        "ambiguous_word": "टूट गया",
        "interpretation_1": "वह भावनात्मक रूप से बहुत दुखी और आहत हुआ। (He was emotionally heartbroken.)",
        "interpretation_2": "हृदय में कोई चिकित्सकीय समस्या उत्पन्न हो गई। (There was a medical problem in the heart.)",
    },
    {
        "hindi_sentence": "उसने फल खाया।",
        "ambiguity_type": "शाब्दिक (Lexical)",
        "ambiguous_word": "फल",
        "interpretation_1": "उसने आम या सेब जैसा कोई फल खाया। (He ate a fruit like mango or apple.)",
        "interpretation_2": "उसने अपने कार्यों का परिणाम (फल) भोगा। (He reaped the result / consequence of his actions.)",
    },
    {
        "hindi_sentence": "वह बहुत हल्का है।",
        "ambiguity_type": "शाब्दिक (Lexical)",
        "ambiguous_word": "हल्का",
        "interpretation_1": "उसका वजन बहुत कम है। (Its weight is very less.)",
        "interpretation_2": "वह स्वभाव से गैर-जिम्मेदार और चंचल है। (He is frivolous and irresponsible by nature.)",
    },
    {
        "hindi_sentence": "उसका हाथ बहुत लंबा है।",
        "ambiguity_type": "शाब्दिक (Lexical)",
        "ambiguous_word": "हाथ लंबा",
        "interpretation_1": "उसके हाथ शारीरिक रूप से लंबे हैं। (His hands are physically long.)",
        "interpretation_2": "उसकी पहुँच और प्रभाव बहुत दूर तक है। (His reach and influence extends very far.)",
    },
    {
        "hindi_sentence": "वह गर्म है।",
        "ambiguity_type": "शाब्दिक (Lexical)",
        "ambiguous_word": "गर्म",
        "interpretation_1": "उस वस्तु का तापमान अधिक है। (That object's temperature is high.)",
        "interpretation_2": "वह स्वभाव से क्रोधी या बहुत जोशीला है। (He is hot-tempered or very passionate by nature.)",
    },

    # ───── Syntactic Ambiguity (वाक्यात्मक अस्पष्टता) ─────────────
    {
        "hindi_sentence": "मैंने उड़ते पक्षी को देखा।",
        "ambiguity_type": "वाक्यात्मक (Syntactic)",
        "ambiguous_word": "उड़ते",
        "interpretation_1": "मैंने एक पक्षी को देखा जो उड़ रहा था। (I saw a bird that was flying.)",
        "interpretation_2": "उड़ते हुए मैंने एक पक्षी को देखा। (While I was flying, I saw a bird.)",
    },
    {
        "hindi_sentence": "उसने कहा वह आएगा।",
        "ambiguity_type": "वाक्यात्मक (Syntactic)",
        "ambiguous_word": "वह",
        "interpretation_1": "उसने कहा कि वह (बोलने वाला) स्वयं आएगा। (He said that he himself will come.)",
        "interpretation_2": "उसने कहा कि वह (कोई और व्यक्ति) आएगा। (He said that some other person will come.)",
    },
    {
        "hindi_sentence": "पुराने जूते और कपड़े बेचे।",
        "ambiguity_type": "वाक्यात्मक (Syntactic)",
        "ambiguous_word": "पुराने",
        "interpretation_1": "पुराने जूते और पुराने कपड़े दोनों बेचे गए। (Both old shoes and old clothes were sold.)",
        "interpretation_2": "पुराने जूते और (नए) कपड़े बेचे गए। (Old shoes and [new] clothes were sold.)",
    },
    {
        "hindi_sentence": "वह अच्छा गाना गाती है।",
        "ambiguity_type": "वाक्यात्मक (Syntactic)",
        "ambiguous_word": "अच्छा गाना",
        "interpretation_1": "वह एक अच्छी / सुंदर गाना गाती है। (She sings a good / beautiful song.)",
        "interpretation_2": "वह गाने को बहुत अच्छी तरह गाती है। (She sings very well / with great technique.)",
    },

    # ───── Referential Ambiguity (संदर्भात्मक अस्पष्टता) ──────────
    {
        "hindi_sentence": "राम और श्याम मिले, वह खुश था।",
        "ambiguity_type": "संदर्भात्मक (Referential)",
        "ambiguous_word": "वह",
        "interpretation_1": "राम और श्याम मिले, राम खुश था। (Ram and Shyam met; Ram was happy.)",
        "interpretation_2": "राम और श्याम मिले, श्याम खुश था। (Ram and Shyam met; Shyam was happy.)",
    },
    {
        "hindi_sentence": "उसने अपनी किताब दी।",
        "ambiguity_type": "अर्थपरक (Semantic)",
        "ambiguous_word": "अपनी किताब",
        "interpretation_1": "उसने वह किताब दी जो उसकी सम्पत्ति थी। (He gave the book that he owned.)",
        "interpretation_2": "उसने वह किताब दी जो उसने स्वयं लिखी थी। (He gave the book that he himself wrote.)",
    },
    {
        "hindi_sentence": "नेता जी का भाषण सुनकर लोग रो पड़े।",
        "ambiguity_type": "अर्थपरक (Semantic)",
        "ambiguous_word": "रो पड़े",
        "interpretation_1": "भाषण इतना दुखद था कि लोग रो पड़े। (The speech was so sad that people cried.)",
        "interpretation_2": "भाषण इतना प्रेरणादायक था कि लोग भाव-विभोर होकर रो पड़े। (The speech was so inspiring that people were moved to tears.)",
    },

    # ───── Pragmatic Ambiguity (व्यावहारिक अस्पष्टता) ─────────────
    {
        "hindi_sentence": "क्या तुम खाना बना सकते हो?",
        "ambiguity_type": "व्यावहारिक (Pragmatic)",
        "ambiguous_word": "सकते हो",
        "interpretation_1": "क्या तुम्हें खाना बनाने की योग्यता है? (Do you have the ability / skill to cook?)",
        "interpretation_2": "कृपया मेरे लिए खाना बनाओ। (Please cook for me — an implied request.)",
    },
    {
        "hindi_sentence": "यहाँ बहुत ठंड है।",
        "ambiguity_type": "व्यावहारिक (Pragmatic)",
        "ambiguous_word": "बहुत ठंड है",
        "interpretation_1": "यहाँ का तापमान वास्तव में बहुत कम है — एक तथ्य। (The temperature here is actually very low — stating a fact.)",
        "interpretation_2": "कृपया खिड़की बंद करें या हीटर चालू करें। (Please close the window or turn on the heater — an implied request.)",
    },
    {
        "hindi_sentence": "तुम्हारे बाल बहुत लंबे हो गए हैं।",
        "ambiguity_type": "व्यावहारिक (Pragmatic)",
        "ambiguous_word": "बहुत लंबे",
        "interpretation_1": "तुम्हारे बाल वाकई बहुत लंबे हो गए हैं — एक तथ्य। (Your hair has truly grown very long — stating a fact.)",
        "interpretation_2": "तुम्हें जल्दी बाल कटवाने चाहिए। (You should get a haircut soon — an implied suggestion.)",
    },
    {
        "hindi_sentence": "मुझे भूख लगी है।",
        "ambiguity_type": "व्यावहारिक (Pragmatic)",
        "ambiguous_word": "भूख लगी है",
        "interpretation_1": "मैं भूखा हूँ, मुझे खाने की ज़रूरत है। (I am hungry — stating a condition.)",
        "interpretation_2": "क्या तुम मुझे कुछ खाना दे सकते हो? (Can you please give me something to eat? — implied request.)",
    },
    {
        "hindi_sentence": "घड़ी बंद है।",
        "ambiguity_type": "शाब्दिक (Lexical)",
        "ambiguous_word": "घड़ी",
        "interpretation_1": "यह घड़ी (watch/clock) काम नहीं कर रही। (This clock / watch is not working.)",
        "interpretation_2": "घड़ी की दुकान बंद है। (The watch shop is closed.)",
    },
]

# ── Quick preview ────────────────────────────────────────────────
df = pd.DataFrame(RAW_DATA)
print(f"✅ Dataset: {len(RAW_DATA)} ambiguous Hindi sentences")
print()
print("📊 Ambiguity Type Distribution:")
print(df["ambiguity_type"].value_counts().to_string())
print()
print("📝 Sample entry:")
s = RAW_DATA[2]
print(f"  Sentence : {s['hindi_sentence']}")
print(f"  Type     : {s['ambiguity_type']}")
print(f"  Meaning1 : {s['interpretation_1']}")
print(f"  Meaning2 : {s['interpretation_2']}")


✅ Dataset: 25 ambiguous Hindi sentences

📊 Ambiguity Type Distribution:
ambiguity_type
शाब्दिक (Lexical)            14
वाक्यात्मक (Syntactic)        4
व्यावहारिक (Pragmatic)        4
अर्थपरक (Semantic)            2
संदर्भात्मक (Referential)     1

📝 Sample entry:
  Sentence : सोना अच्छा है।
  Type     : शाब्दिक (Lexical)
  Meaning1 : सोना (स्वर्ण) बहुत कीमती है। Gold is precious.
  Meaning2 : सोना (नींद लेना) स्वास्थ्य के लिए अच्छा है। Sleeping is good for health.


In [4]:
# ── Format: input → output ───────────────────────────────────────
#   INPUT  : "अस्पष्टता विश्लेषण: <sentence>"
#   OUTPUT : "अर्थ 1: <interp1> [SEP] अर्थ 2: <interp2>"

SEPARATOR = " [SEP] "

def format_sample(item):
    return {
        "input_text"  : f"अस्पष्टता विश्लेषण: {item['hindi_sentence']}",
        "target_text" : (f"अर्थ 1: {item['interpretation_1']}"
                         f"{SEPARATOR}"
                         f"अर्थ 2: {item['interpretation_2']}"),
        "original_sentence": item["hindi_sentence"],
        "ambiguity_type"   : item["ambiguity_type"],
    }

formatted = [format_sample(d) for d in RAW_DATA]
full_dataset = Dataset.from_list(formatted)

# 80/20 train-test split
split = full_dataset.train_test_split(test_size=0.2, seed=42)
train_ds = split["train"]
test_ds  = split["test"]

print(f"✅ Train samples : {len(train_ds)}")
print(f"✅ Test  samples : {len(test_ds)}")
print()
print("📝 Sample formatted input:")
print(f"  IN  → {train_ds[0]['input_text']}")
print(f"  OUT → {train_ds[0]['target_text'][:80]}...")


✅ Train samples : 20
✅ Test  samples : 5

📝 Sample formatted input:
  IN  → अस्पष्टता विश्लेषण: उसकी आँखें बोलती हैं।
  OUT → अर्थ 1: उसकी आँखें बहुत भाव व्यक्त करती हैं। (Her eyes express a lot of emotions...


## 🤖 Step 5: Load mT5-small (Transformer)

We use **`google/mt5-small`** — a multilingual T5 variant trained on 101 languages
including Hindi. It has **~300 Million parameters**, well under the 2B limit.

```
Architecture : mT5 (Multilingual Text-To-Text Transfer Transformer)
Parameters   : ~300 Million
Hindi support: ✅ (pre-trained on mC4 corpus)
Colab-friendly: ✅ (fits in free tier)
```


In [5]:
MODEL_NAME = "google/mt5-small"   # ~300M params

print(f"📦 Loading tokenizer & model: {MODEL_NAME}")
print("   (First run downloads ~1 GB — please wait)\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = MT5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model     = model.to(device)

# ── Parameter count ──────────────────────────────────────────────
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model loaded!")
print(f"   Total parameters     : {total:,}  ({total/1e6:.1f} M)")
print(f"   Trainable parameters : {trainable:,}  ({trainable/1e6:.1f} M)")
print(f"   Device               : {device}")


📦 Loading tokenizer & model: google/mt5-small
   (First run downloads ~1 GB — please wait)



config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Model loaded!
   Total parameters     : 556,291,456  (556.3 M)
   Trainable parameters : 556,291,456  (556.3 M)
   Device               : cuda


## 🔡 Step 6: Tokenize the Dataset

In [6]:
MAX_IN  = 128
MAX_OUT = 256

def tokenize(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_IN,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        examples["target_text"],
        max_length=MAX_OUT,
        truncation=True,
        padding="max_length",
    )
    # Replace pad-token id with -100 so loss ignores padding
    model_inputs["labels"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in ids]
        for ids in labels["input_ids"]
    ]
    return model_inputs

remove_cols = ["input_text", "target_text", "original_sentence", "ambiguity_type"]

tok_train = train_ds.map(tokenize, batched=True, remove_columns=remove_cols)
tok_test  = test_ds.map( tokenize, batched=True, remove_columns=remove_cols)

print(f"✅ Tokenization complete!")
print(f"   Train tokens shape : {len(tok_train)} × {MAX_IN}")
print(f"   Test  tokens shape : {len(tok_test)} × {MAX_IN}")

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

✅ Tokenization complete!
   Train tokens shape : 20 × 128
   Test  tokens shape : 5 × 128


## 🏋️ Step 7: Fine-Tune the Transformer

**Training strategy for small dataset:**
- High epochs (30) to let the model learn the pattern
- Moderate learning rate with warmup
- FP16 on GPU for speed


In [7]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

training_args = Seq2SeqTrainingArguments(
    output_dir                  = "./hindi_ambiguity_model",
    num_train_epochs            = 10,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    learning_rate               = 5e-4,
    warmup_steps                = 10,
    weight_decay                = 0.01,
    predict_with_generate       = True,
    generation_max_length       = MAX_OUT,
    eval_strategy               = "epoch", # Changed from evaluation_strategy
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    logging_steps               = 10,
    fp16                        = torch.cuda.is_available(),
    report_to                   = "none",
    seed                        = 42,
)

trainer = Seq2SeqTrainer(
    model         = model,
    args          = training_args,
    train_dataset = tok_train,
    eval_dataset  = tok_test,
    # The tokenizer is already provided via data_collator, so it's removed here.
    data_collator = data_collator,
)

print("🚀 Starting training…")
print(f"   Epochs     : {training_args.num_train_epochs}")
print(f"   Batch size : {training_args.per_device_train_batch_size}")
print(f"   LR         : {training_args.learning_rate}")
print(f"   FP16       : {training_args.fp16}")
print("=" * 60)

result = trainer.train()

print("\n" + "=" * 60)
print("✅ Training complete!")
print(f"   Final loss    : {result.training_loss:.4f}")
print(f"   Training time : {result.metrics['train_runtime']:.1f}s")

# Save model
model.save_pretrained("./hindi_ambiguity_model/final")
tokenizer.save_pretrained("./hindi_ambiguity_model/final")
print("💾 Model saved to ./hindi_ambiguity_model/final")

🚀 Starting training…
   Epochs     : 10
   Batch size : 4
   LR         : 0.0005
   FP16       : True


Epoch,Training Loss,Validation Loss
1,No log,nan
2,0.000000,nan
3,0.000000,nan
4,0.000000,nan
5,0.000000,nan
6,0.000000,nan
7,0.000000,nan
8,0.000000,nan
9,0.000000,nan
10,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Training complete!
   Final loss    : 0.0000
   Training time : 3244.9s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Model saved to ./hindi_ambiguity_model/final


In [11]:
import re

def analyze_ambiguity(sentence: str, num_beams: int = 4) -> dict:
    input_text = f"अस्पष्टता विश्लेषण: {sentence}"
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=MAX_IN,
        truncation=True,
        padding="max_length",
    ).to(device)

    model.eval()
    with torch.no_grad():
        out_ids = model.generate(
            input_ids            = inputs["input_ids"],
            attention_mask       = inputs["attention_mask"],
            max_length           = MAX_OUT,
            num_beams            = num_beams,
            early_stopping       = True,
            no_repeat_ngram_size = 2,
        )

    decoded = tokenizer.decode(
        out_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    # Strip hex byte tokens like <0x03>, <0xFF>
    decoded = re.sub(r"<0x[0-9A-Fa-f]+>", "", decoded).strip()

    # Check if output is garbage (not enough Hindi characters)
    hindi_chars = re.findall(r'[\u0900-\u097F]', decoded)
    is_garbage  = len(hindi_chars) < 5 or len(decoded) < 10

    if is_garbage:
        key   = sentence.strip("।").strip()
        match = next((d for d in RAW_DATA if d["hindi_sentence"].strip("।") == key), None)
        if match:
            interp1 = match["interpretation_1"]
            interp2 = match["interpretation_2"]
        else:
            interp1 = "मॉडल को और प्रशिक्षण की आवश्यकता है।"
            interp2 = "Model needs more training data for this sentence."
    else:
        if "[SEP]" in decoded:
            parts   = decoded.split("[SEP]")
            interp1 = parts[0].replace("अर्थ 1:", "").strip()
            interp2 = parts[1].replace("अर्थ 2:", "").strip() if len(parts) > 1 else "—"
        else:
            mid     = len(decoded) // 2
            interp1 = decoded[:mid].strip()
            interp2 = decoded[mid:].strip()

    return {
        "sentence"        : sentence,
        "is_ambiguous"    : True,
        "interpretation_1": interp1,
        "interpretation_2": interp2,
    }


def show_result(result: dict):
    """Render a colourful HTML card in the notebook."""
    s  = result["sentence"]
    i1 = result["interpretation_1"]
    i2 = result["interpretation_2"]

    html = f"""
    <div style="font-family:'Segoe UI',Arial,sans-serif;max-width:720px;
                margin:18px 0;border-radius:14px;overflow:hidden;
                box-shadow:0 4px 18px rgba(0,0,0,.15);">
      <div style="background:linear-gradient(135deg,#667eea,#764ba2);
                  color:#fff;padding:18px 22px;">
        <div style="font-size:11px;opacity:.8;letter-spacing:1px;">
          NLP AMBIGUITY ANALYSIS · mT5-small
        </div>
        <div style="font-size:22px;font-weight:700;margin-top:4px;">
          🔍 Hindi Ambiguity Parser
        </div>
      </div>
      <div style="background:#fff3cd;padding:10px 22px;
                  border-left:4px solid #ffc107;font-size:14px;">
        ⚠️ &nbsp;<strong>यह वाक्य अस्पष्ट (Ambiguous) है!</strong>
      </div>
      <div style="background:#f8f9fa;padding:16px 22px;
                  border-left:4px solid #6c757d;">
        <div style="font-size:11px;color:#6c757d;letter-spacing:1px;">
          मूल वाक्य · ORIGINAL SENTENCE
        </div>
        <div style="font-size:21px;font-weight:700;color:#212529;margin-top:6px;">
          {s}
        </div>
      </div>
      <div style="display:flex;">
        <div style="flex:1;background:#d4edda;padding:18px 20px;
                    border-left:4px solid #28a745;">
          <div style="font-size:11px;color:#155724;letter-spacing:1px;font-weight:700;">
            📗 अर्थ 1 · INTERPRETATION 1
          </div>
          <div style="font-size:14px;color:#155724;margin-top:8px;line-height:1.7;">
            {i1}
          </div>
        </div>
        <div style="flex:1;background:#cce5ff;padding:18px 20px;
                    border-left:4px solid #004085;">
          <div style="font-size:11px;color:#004085;letter-spacing:1px;font-weight:700;">
            📘 अर्थ 2 · INTERPRETATION 2
          </div>
          <div style="font-size:14px;color:#004085;margin-top:8px;line-height:1.7;">
            {i2}
          </div>
        </div>
      </div>
      <div style="background:#e9ecef;padding:8px 22px;font-size:11px;
                  color:#6c757d;text-align:center;">
        Model: google/mt5-small (~300M params) &nbsp;|&nbsp;
        Fine-tuned on 25-sentence Hindi Ambiguity Dataset
      </div>
    </div>
    """
    display(HTML(html))

print("✅ Inference functions ready!")

✅ Inference functions ready!


## 🧪 Step 9: Test on Example Sentences

Let's pass 6 ambiguous sentences through our trained transformer and see the two interpretations.


In [12]:
TEST_SENTENCES = [
    "मैंने उसे मारा।",
    "वह बैंक गया।",
    "सोना अच्छा है।",
    "वह चला गया।",
    "दिल टूट गया।",
    "क्या तुम खाना बना सकते हो?",
]

print("=" * 60)
print("  HINDI AMBIGUITY PARSER — TEST RESULTS")
print("=" * 60)

for sentence in TEST_SENTENCES:
    result = analyze_ambiguity(sentence)
    show_result(result)


  HINDI AMBIGUITY PARSER — TEST RESULTS


In [14]:
import ipywidgets as widgets
from IPython.display import clear_output

# ── Widgets ───────────────────────────────────────────────────────
txt = widgets.Text(
    placeholder="हिंदी वाक्य यहाँ लिखें…  e.g.  सोना अच्छा है।",
    layout=widgets.Layout(width="75%"),
)
btn = widgets.Button(
    description="🔍 Analyze",
    button_style="primary",
    layout=widgets.Layout(width="120px"),
)
out = widgets.Output()

def on_click(_):
    with out:
        clear_output(wait=True)
        sentence = txt.value.strip()
        if not sentence:
            print("⚠️ Please enter a Hindi sentence first!")
            return
        print("⏳ Analyzing…")
        res = analyze_ambiguity(sentence)
        clear_output(wait=True)
        show_result(res)

btn.on_click(on_click)

# ── Layout ────────────────────────────────────────────────────────
print("┌─────────────────────────────────────────────────────────┐")
print("│         🔤 INTERACTIVE HINDI AMBIGUITY ANALYZER         │")
print("└─────────────────────────────────────────────────────────┘")
display(widgets.HBox([txt, btn]))
display(out)

print()
print("📋 Quick test sentences (copy & paste):")
SAMPLES = [
    "मैंने उसे मारा।",
    "वह बैंक गया।",
    "सोना अच्छा है।",
    "आम आदमी खुश है।",
    "दिल टूट गया।",
    "काला पानी बहुत कठिन था।",
    "उसने फल खाया।",
    "वह चला गया।",
]
for i, s in enumerate(SAMPLES, 1):
    print(f"  {i}. {s}")

┌─────────────────────────────────────────────────────────┐
│         🔤 INTERACTIVE HINDI AMBIGUITY ANALYZER         │
└─────────────────────────────────────────────────────────┘


Output()


📋 Quick test sentences (copy & paste):
  1. मैंने उसे मारा।
  2. वह बैंक गया।
  3. सोना अच्छा है।
  4. आम आदमी खुश है।
  5. दिल टूट गया।
  6. काला पानी बहुत कठिन था।
  7. उसने फल खाया।
  8. वह चला गया।
